# ONNX for Generative AI — Hands-On Practice

This notebook provides exercises exploring generative AI concepts
with ONNX Runtime, focusing on memory estimation, pipeline orchestration,
and scheduling patterns.

## Exercise 1: Diffusion Pipeline Scheduler Simulation

Simulate the iterative denoising loop used in Stable Diffusion.

In [ ]:
import numpy as np
import time


def simulate_diffusion_pipeline(
    num_steps: int = 20,
    latent_shape: tuple = (1, 4, 64, 64),
) -> dict:
    """Simulate a diffusion pipeline's scheduling loop."""
    # Initialize latent noise
    latent = np.random.randn(*latent_shape).astype(np.float32)

    timesteps = np.linspace(999, 0, num_steps, dtype=np.float32)
    step_times = []

    for i, t in enumerate(timesteps):
        t0 = time.perf_counter()

        # Simulate UNet forward pass (in reality: sess.run on UNet ONNX)
        noise_pred = np.random.randn(*latent_shape).astype(np.float32) * 0.01

        # Scheduler step: x_{t-1} = f(x_t, noise_pred, t)
        alpha = 1.0 - (t / 1000.0)
        latent = alpha * latent + (1 - alpha) * noise_pred

        step_times.append((time.perf_counter() - t0) * 1000)

    # Simulate VAE decode
    image = np.clip(latent[:, :3, :, :] * 0.5 + 0.5, 0, 1)

    return {
        "num_steps": num_steps,
        "total_time_ms": sum(step_times),
        "avg_step_ms": np.mean(step_times),
        "output_shape": image.shape,
        "latent_memory_mb": latent.nbytes / 1e6,
    }


result = simulate_diffusion_pipeline(num_steps=20)
print(f"Steps: {result['num_steps']}")
print(f"Total time: {result['total_time_ms']:.1f}ms")
print(f"Avg step: {result['avg_step_ms']:.2f}ms")
print(f"Latent memory: {result['latent_memory_mb']:.2f}MB")

## Exercise 2: KV-Cache Memory Estimator

Estimate KV-cache memory for LLM inference at various sequence lengths.

In [ ]:
import numpy as np


def estimate_kv_cache_memory(
    num_layers: int,
    num_heads: int,
    head_dim: int,
    seq_length: int,
    batch_size: int = 1,
    precision: str = "fp16",
) -> dict:
    """Estimate KV-cache memory for transformer LLM inference."""
    bytes_per_element = {"fp32": 4, "fp16": 2, "int8": 1}.get(precision, 2)

    # KV-cache: 2 (K+V) × layers × batch × heads × seq × head_dim
    kv_elements = 2 * num_layers * batch_size * num_heads * seq_length * head_dim
    kv_bytes = kv_elements * bytes_per_element

    return {
        "num_layers": num_layers,
        "seq_length": seq_length,
        "precision": precision,
        "kv_cache_gb": round(kv_bytes / 1e9, 3),
        "kv_cache_mb": round(kv_bytes / 1e6, 1),
    }


# Compare KV-cache sizes for different model scales
configs = [
    {"name": "GPT-2 (124M)", "num_layers": 12, "num_heads": 12, "head_dim": 64},
    {"name": "LLaMA-7B", "num_layers": 32, "num_heads": 32, "head_dim": 128},
    {"name": "LLaMA-70B", "num_layers": 80, "num_heads": 64, "head_dim": 128},
]

for cfg in configs:
    for seq_len in [512, 2048, 8192]:
        est = estimate_kv_cache_memory(
            cfg["num_layers"], cfg["num_heads"], cfg["head_dim"],
            seq_len, precision="fp16"
        )
        print(f"  {cfg['name']} @ seq={seq_len}: KV-cache = {est['kv_cache_mb']}MB")
    print()

## Exercise 3: Precision Comparison

Compare memory and theoretical throughput across precision levels.

In [ ]:
import numpy as np


def precision_comparison(num_params_millions: float) -> list:
    """Compare different precision formats for a given model size."""
    precisions = {
        "FP32": {"bytes": 4, "relative_speed": 1.0},
        "FP16": {"bytes": 2, "relative_speed": 1.8},
        "INT8": {"bytes": 1, "relative_speed": 2.5},
        "INT4": {"bytes": 0.5, "relative_speed": 3.0},
    }

    results = []
    num_params = num_params_millions * 1e6
    for name, info in precisions.items():
        size_gb = (num_params * info["bytes"]) / 1e9
        results.append({
            "precision": name,
            "model_size_gb": round(size_gb, 2),
            "relative_speed": info["relative_speed"],
            "memory_savings": f"{(1 - info['bytes']/4)*100:.0f}%",
        })

    return results


# Compare for a 7B parameter model
print("7B parameter model precision comparison:")
for r in precision_comparison(7000):
    print(f"  {r['precision']}: {r['model_size_gb']}GB, "
          f"~{r['relative_speed']}x speed, {r['memory_savings']} memory saved")

## Exercise 4: Multi-Model Pipeline Orchestrator

Build a simple pipeline orchestrator for multi-ONNX-model workflows.

In [ ]:
import time
from typing import Dict, List


class PipelineStage:
    """Represents one ONNX model stage in a generative pipeline."""

    def __init__(self, name: str, estimated_ms: float, memory_mb: float):
        self.name = name
        self.estimated_ms = estimated_ms
        self.memory_mb = memory_mb

    def run(self, inputs: dict) -> dict:
        """Simulate running this pipeline stage."""
        time.sleep(self.estimated_ms / 1000)
        return {"stage": self.name, "status": "completed"}


class GenerativePipeline:
    """Orchestrate multiple ONNX model stages."""

    def __init__(self, stages: List[PipelineStage]):
        self.stages = stages

    def estimate_resources(self) -> dict:
        total_time = sum(s.estimated_ms for s in self.stages)
        peak_memory = max(s.memory_mb for s in self.stages)
        total_memory = sum(s.memory_mb for s in self.stages)
        return {
            "total_estimated_ms": total_time,
            "peak_memory_mb": peak_memory,
            "total_memory_mb": total_memory,
            "stages": len(self.stages),
        }


# Simulate a Stable Diffusion pipeline
pipeline = GenerativePipeline([
    PipelineStage("CLIP text encoder", estimated_ms=15, memory_mb=500),
    PipelineStage("UNet (20 steps)", estimated_ms=2000, memory_mb=3400),
    PipelineStage("VAE decode", estimated_ms=50, memory_mb=1200),
])

resources = pipeline.estimate_resources()
print("Stable Diffusion pipeline estimate:")
for k, v in resources.items():
    print(f"  {k}: {v}")

## Summary

In this notebook you practiced:
- Simulating diffusion pipeline scheduling loops
- Estimating KV-cache memory for LLM inference at different scales
- Comparing precision formats for memory and throughput trade-offs
- Building multi-model pipeline orchestrators for generative AI workflows